# 🏡 Land Price Prediction — Decision Tree Regressor

This notebook builds a machine learning model to **predict land prices per square meter** using parcel-level features such as location, road access, land use, and nearby facilities.

**Dataset:** 147 real estate parcels from the West Bank  
**Target:** `actual_price_per_m2` (continuous)  
**Model:** Decision Tree Regressor with GridSearchCV tuning  

---
**Pipeline Overview:**
1. Data Loading & Filtering
2. Data Cleaning
3. Feature Engineering
4. Model Training & Hyperparameter Tuning
5. Evaluation
6. Model Export

## ⚙️ Setup & Dependencies

In [ ]:
!pip install --upgrade scikit-learn

In [ ]:
import sklearn
print(sklearn.__version__)

## 📂 Data Loading

Load the parcels dataset and apply an initial filter to exclude incomplete or irrelevant area entries.

In [ ]:
import pandas as pd

df = pd.read_csv("/content/parcelsData.csv", nrows=147)

# Exclude rows where Area column equals "Yaqeen"
df = df[df['Area'] != 'Yaqeen']
df = df.reset_index(drop=True)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

df

## 🧹 Data Cleaning

- Drop columns not relevant to price prediction (identifiers, redundant fields)
- Standardize boolean/binary features by filling missing values with `0`
- Validate and normalize road status fields against a known set of valid categories
- Derive road width values based on road status (zero-fill widths where no road exists)

In [ ]:
import numpy as np

# Drop columns not useful for modeling
data = df.drop(columns=[
    "project_name",
    "description",
    "parcel_no",
    "neighborhood_no",
    "Governorate",
    "actual_price_per_m2",
    "land_use_types",
    "land_type",
    "Town",
    "ownership_document_type"
])

data = data.replace("", np.nan)

# Boolean fields → fill missing with 0
boolean_columns = [
    "land_use_residential",
    "land_use_commercial",
    "land_use_agricultural",
    "land_use_industrial",
    "FACTORIES_NEARBY",
    "NOISY_FACILITIES",
    "ANIMAL_FARMS",
    "hospitals_facility",
    "schools_facility",
    "police_facility",
    "municipality_facility",
]

for col in boolean_columns:
    if col in data.columns:
        data[col] = data[col].fillna(0).astype(int)

# Validate road status values against known categories
VALID_ROAD_STATUSES = {
    "PUBLIC_EXISTING_UNPAVED",
    "PUBLIC_EXISTING_PAVED",
    "PRIVATE_EXISTING_UNPAVED",
    "PRIVATE_EXISTING_PAVED",
    "PRIVATE_PROPOSED_UNPAVED",
    "PUBLIC_PROPOSED_UNPAVED",
    "FALSE",
}

road_status_columns = ["road_status1", "road_status2", "road_status3"]

for col in road_status_columns:
    if col in data.columns:
        data[col] = (
            data[col]
            .astype(str)
            .str.strip()
            .str.upper()
            .replace("NAN", np.nan)
        )

for col in road_status_columns:
    if col in data.columns:
        invalid_mask = (
            data[col].notna() &
            ~data[col].isin(VALID_ROAD_STATUSES)
        )
        data = data[~invalid_mask]

# Set road width to 0 where no road exists
width_columns = ["width_m", "width_m.1", "width_m.2"]

for i, width_col in enumerate(width_columns, start=1):
    status_col = f"road_status{i}"
    if width_col in data.columns and status_col in data.columns:
        data.loc[
            data[status_col].isna() | (data[status_col] == "FALSE"),
            width_col
        ] = 0
        data[width_col] = data[width_col].fillna(0)

data = data.reset_index(drop=True)
data

## 🔧 Feature Engineering

- Separate the target variable (`actual_price_per_m2`) from the feature matrix
- Encode the binary `water` column (YES/NO → 1/0)
- Define feature groups: **categorical**, **numeric**, and **binary** — used later by the preprocessing pipeline

In [ ]:
# Separate target and features
y = df.loc[data.index, "actual_price_per_m2"]
y = y.reset_index(drop=True)
data = data.reset_index(drop=True)
X = data

# Sanity check: ensure X and y are aligned
assert len(X) == len(y)
assert X.index.equals(y.index)

In [ ]:
# Encode binary water field
data['water'] = data['water'].map({'YES': 1, 'NO': 0})

# Define feature groups for the preprocessing pipeline
categorical_cols = [
    "Area",
    "Neighborhood",
    "political_classification",
    "parcel_shape",
    "road_status1",
    "road_status2",
    "road_status3",
    "slope",
    "view_quality",
    "electricity",
    "Sewage",
]

numeric_cols = [
    "area_m2",
    "parcel_frontage (m)",
    "width_m",
    "width_m.1",
    "width_m.2",
]

binary_cols = [
    "land_use_residential",
    "land_use_commercial",
    "land_use_agricultural",
    "land_use_industrial",
    "FACTORIES_NEARBY",
    "NOISY_FACILITIES",
    "ANIMAL_FARMS",
    "hospitals_facility",
    "schools_facility",
    "police_facility",
    "municipality_facility",
    "water",
]

## 🤖 Model Training & Hyperparameter Tuning

A `Pipeline` is constructed with:
- **Preprocessing:** `StandardScaler` for numeric features, `OneHotEncoder` for categoricals, passthrough for binary flags
- **Model:** `DecisionTreeRegressor`

`GridSearchCV` with 5-fold cross-validation is used to find the optimal combination of `max_depth`, `min_samples_leaf`, and `min_samples_split`.

In [ ]:
from sklearn.tree import DecisionTreeRegressor
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
import pandas as pd

# Preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_cols),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("bin", "passthrough", binary_cols),
    ]
)

# Full model pipeline
model_pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", DecisionTreeRegressor(random_state=42))
])

# Hyperparameter grid
param_grid = {
    'regressor__max_depth': [3, 4, 6],
    'regressor__min_samples_leaf': [3, 5, 7, 10, 12, 14, 16],
    'regressor__min_samples_split': [5, 8, 10, 12, 14]
}

print("=" * 50)
print("GRID SEARCH — Finding Best Hyperparameters")
print("=" * 50)

grid_search = GridSearchCV(
    model_pipeline,
    param_grid,
    cv=5,
    scoring='r2',
    verbose=1,
    return_train_score=True,
    n_jobs=-1
)

grid_search.fit(X, y)

print(f"\n✅ Best parameters: {grid_search.best_params_}")
print(f"✅ Best CV R²: {grid_search.best_score_:.3f}")

# Use the best estimator going forward
model_pipeline = grid_search.best_estimator_

# Show top 5 parameter combinations
results_df = pd.DataFrame(grid_search.cv_results_)
top_5 = results_df.nlargest(5, 'mean_test_score')[
    ['params', 'mean_test_score', 'std_test_score', 'mean_train_score']
]
print("\nTop 5 Parameter Combinations:")
for idx, row in top_5.iterrows():
    print(f"  {row['params']}")
    print(f"    CV R²: {row['mean_test_score']:.3f} (±{row['std_test_score']:.3f}), Train R²: {row['mean_train_score']:.3f}\n")

print("=" * 50)

## 📊 Evaluation

The tuned model is evaluated using:
- **5-fold cross-validation** with R², MAE, MSE, and RMSE
- **Actual vs. Predicted** scatter plots (full dataset and train/test split)
- **Decision tree visualization** to interpret the model's learned structure

In [ ]:
from sklearn.model_selection import cross_validate
import numpy as np

print("=" * 50)
print("FINAL MODEL EVALUATION — 5-Fold Cross-Validation")
print(f"Best Parameters: {grid_search.best_params_}")
print("=" * 50)

cv_results = cross_validate(
    model_pipeline, X, y,
    cv=5,
    scoring=['r2', 'neg_mean_absolute_error', 'neg_root_mean_squared_error', 'neg_mean_squared_error'],
    return_train_score=True
)

train_r2 = cv_results['train_r2'].mean()
test_r2  = cv_results['test_r2'].mean()
test_r2_std = cv_results['test_r2'].std()
test_mae  = -cv_results['test_neg_mean_absolute_error'].mean()
test_mse  = -cv_results['test_neg_mean_squared_error'].mean()
test_rmse = -cv_results['test_neg_root_mean_squared_error'].mean()

print(f"\nR²   — Train: {train_r2:.3f} | Test: {test_r2:.3f} (±{test_r2_std:.3f}) | Gap: {train_r2 - test_r2:.3f}")
print(f"MAE  — {test_mae:.2f}")
print(f"MSE  — {test_mse:.2f}")
print(f"RMSE — {test_rmse:.2f}")

print("\nPer-Fold Results:")
for i in range(5):
    print(f"  Fold {i+1}: Train R²={cv_results['train_r2'][i]:.3f}, Test R²={cv_results['test_r2'][i]:.3f}, "
          f"MAE={-cv_results['test_neg_mean_absolute_error'][i]:.2f}, "
          f"RMSE={-cv_results['test_neg_root_mean_squared_error'][i]:.2f}")

print("=" * 50)

In [ ]:
from sklearn.model_selection import cross_val_score

# Summary MAE via cross-validation
scores = cross_val_score(model_pipeline, X, y, cv=5, scoring="neg_mean_absolute_error")
print(f"Cross-Validated MAE: {-scores.mean():.2f} (±{scores.std():.2f})")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.tree import plot_tree

tree_model = model_pipeline.named_steps['regressor']

# Build feature name list matching the pipeline's output order
feature_names = list(numeric_cols)
cat_transformer = model_pipeline.named_steps['preprocessor'].named_transformers_['cat']
feature_names.extend(cat_transformer.get_feature_names_out(categorical_cols))
feature_names.extend(binary_cols)

plt.figure(figsize=(25, 15))
plot_tree(
    tree_model,
    feature_names=feature_names,
    filled=True,
    rounded=True,
    fontsize=10,
    max_depth=4
)
plt.title("Decision Tree Visualization (max depth = 4)", fontsize=16)
plt.tight_layout()
plt.savefig('decision_tree.png', dpi=300, bbox_inches='tight')
plt.show()
print("Tree saved as 'decision_tree.png'")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import r2_score

# Cross-validated predictions (unbiased estimate of generalization)
y_pred = cross_val_predict(model_pipeline, X, y, cv=5)
r2 = r2_score(y, y_pred)

plt.figure(figsize=(10, 8))
plt.scatter(y, y_pred, alpha=0.6, edgecolors='k', s=80)
min_val, max_val = min(y.min(), y_pred.min()), max(y.max(), y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', lw=2, label='Perfect Prediction')
plt.xlabel('Actual Land Price (₪/m²)', fontsize=12)
plt.ylabel('Predicted Land Price (₪/m²)', fontsize=12)
plt.title(f'Actual vs. Predicted Land Prices (CV)\nR² = {r2:.3f}', fontsize=14, fontweight='bold')
plt.legend(fontsize=10)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('r2_plot.png', dpi=300, bbox_inches='tight')
plt.show()
print(f"R² Score: {r2:.3f} | Plot saved as 'r2_plot.png'")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model_pipeline.fit(X_train, y_train)

y_train_pred = model_pipeline.predict(X_train)
y_test_pred  = model_pipeline.predict(X_test)
train_r2 = r2_score(y_train, y_train_pred)
test_r2  = r2_score(y_test,  y_test_pred)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, y_true, y_p, label, color in [
    (axes[0], y_train, y_train_pred, f'Training Set\nR² = {train_r2:.3f}', 'blue'),
    (axes[1], y_test,  y_test_pred,  f'Test Set\nR² = {test_r2:.3f}',      'green'),
]:
    ax.scatter(y_true, y_p, alpha=0.6, edgecolors='k', s=80, color=color)
    lo, hi = min(y_true.min(), y_p.min()), max(y_true.max(), y_p.max())
    ax.plot([lo, hi], [lo, hi], 'r--', lw=2)
    ax.set_xlabel('Actual Land Price (₪/m²)', fontsize=12)
    ax.set_ylabel('Predicted Land Price (₪/m²)', fontsize=12)
    ax.set_title(label, fontsize=13, fontweight='bold')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('r2_train_test_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Training R²: {train_r2:.3f} | Test R²: {test_r2:.3f} | Gap: {train_r2 - test_r2:.3f}")

## 💾 Model Export

Retrain the final model on the **full dataset** (all available samples) using the best hyperparameters found during grid search, then serialize it with `joblib` for deployment or inference.

In [ ]:
import joblib

print(f"Training final model on {len(X)} samples...")
print(f"Parameters: {model_pipeline.named_steps['regressor'].get_params()}")

model_pipeline.fit(X, y)

joblib.dump(model_pipeline, "land_price_model.pkl")

final_r2 = model_pipeline.score(X, y)
print(f"\n✅ Model saved as 'land_price_model.pkl'")
print(f"✅ Final training R² (full dataset): {final_r2:.3f}")